
# Vision RAG using Groq + Gradio

This notebook builds a simple Vision-RAG style pipeline:
- Accepts an image
- Extracts visual description
- Uses Groq LLM to generate a **~10 word explanation**
- Deploys via Gradio

You can enter your **Groq API Key** during runtime.


In [6]:

!pip install groq gradio pillow


In [7]:

from groq import Groq
from PIL import Image
import gradio as gr
import base64
import io


In [8]:

def encode_image(image):
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


In [9]:
def vision_rag(image, api_key):
    try:
        if image is None:
            return "Please upload an image."

        client = Groq(api_key=api_key)

        img_base64 = encode_image(image)

        response = client.chat.completions.create(
            model="llama-3.2-11b-vision-preview",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Describe this image in about 10 words only."},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{img_base64}"
                            },
                        },
                    ],
                }
            ],
        )

        return response.choices[0].message.content

    except Exception as e:
        return str(e)

In [10]:

def app():
    with gr.Blocks() as demo:
        gr.Markdown("## Vision RAG (Groq) - 10 Word Image Explainer")

        api_key = gr.Textbox(label="Enter Groq API Key", type="password")
        image = gr.Image(type="pil")
        output = gr.Textbox(label="Explanation")

        btn = gr.Button("Generate")

        btn.click(vision_rag, inputs=[image, api_key], outputs=output)

    return demo

demo = app()
demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2caaa24e771f9b8a83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
